In [ ]:
# ============================================================
# CELL 1: GPU Check & Environment Setup
# ============================================================
import torch
import os

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Total VRAM   : {total_mem:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️  No GPU found — using CPU (training will be slow)")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, random, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("✅ All imports successful.")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

# ── Paths ──────────────────────────────────────────────────
DATASET_PATH = Path("/kaggle/input/datasets/bapdesilva/betel-dataset-bp/Betel Leaf Image Dataset from Bangladesh/Augmented Images/Train")
AUGMENTED_PATH = Path("/kaggle/working/augmented_dataset")   # offline-augmented copy
FINAL_PATH     = Path("/kaggle/working/final_dataset")        # train/val split lives here

# ── Dataset ────────────────────────────────────────────────
TARGET_PER_CLASS = 2000   # augment up to this count before any training
VAL_SPLIT        = 0.20   # 80/20 train-val split
IMG_SIZE         = 224    # EfficientNetB0 native resolution

# ── Training hyper-parameters ──────────────────────────────
BATCH_SIZE      = 32
NUM_EPOCHS      = 30
LR_HEAD         = 1e-3    # Phase-1: only classifier head
LR_FINE         = 1e-4    # Phase-2: fine-tune unlocked layers
WEIGHT_DECAY    = 1e-4
PATIENCE        = 7       # early-stopping patience (per phase)
UNFREEZE_PCT    = 0.30    # fraction of backbone layers to unlock in Phase-2

CLASS_NAMES = ["Bacterial Leaf Disease", "Dried Leaf",
               "Fungal Brown Spot Disease", "Healthy Leaf"]
NUM_CLASSES = len(CLASS_NAMES)

print("✅ Configuration set.")
print(f"   Dataset path  : {DATASET_PATH}")
print(f"   Target/class  : {TARGET_PER_CLASS}")
print(f"   Val split     : {VAL_SPLIT*100:.0f}%")
print(f"   Device        : {DEVICE}")

In [ ]:
# ============================================================
# CELL 4: Offline Augmentation (run ONCE — before training)
# Creates a balanced dataset at AUGMENTED_PATH
# ============================================================

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

def augment_class_folder(src_folder: Path, dst_folder: Path, target: int):
    """
    Copy originals + generate synthetic images until we reach `target`.
    dst_folder is created fresh each run (idempotent).
    """
    dst_folder.mkdir(parents=True, exist_ok=True)

    # Collect original images
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
    originals = [p for p in src_folder.iterdir() if p.suffix.lower() in exts]

    if not originals:
        print(f"  ⚠️  No images found in {src_folder} — skipping.")
        return 0

    count = 0

    # 1. Copy originals first
    for p in originals:
        shutil.copy(p, dst_folder / p.name)
        count += 1

    # 2. Augment until target is reached
    idx = 0
    while count < target:
        src_img_path = originals[idx % len(originals)]
        img = Image.open(src_img_path).convert("RGB")
        aug_img = aug_transform(img)
        save_name = f"aug_{count:05d}.jpg"
        aug_img.save(dst_folder / save_name)
        count += 1
        idx += 1

    return count


print("=" * 55)
print("OFFLINE AUGMENTATION  (target = {} images/class)".format(TARGET_PER_CLASS))
print("=" * 55)

total_generated = 0
for cls in CLASS_NAMES:
    src = DATASET_PATH / cls
    dst = AUGMENTED_PATH / cls

    if not src.exists():
        print(f"  ❌ Folder not found: {src}")
        continue

    n_orig = len([f for f in src.iterdir()
                  if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tiff",".webp"}])
    print(f"\n  [{cls}]  originals: {n_orig}", end=" → ")

    n_final = augment_class_folder(src, dst, TARGET_PER_CLASS)
    print(f"total after augmentation: {n_final}")
    total_generated += n_final

print(f"\n✅ Augmentation complete.  Grand total images: {total_generated}")